# Pretext Multi-Task CNN Training for Out-Of-Distribution Plant Disease Identification

This notebook trains the **Pretext Multi-Task CNN** model (`train/train_pretext_multitask.py`) with self-supervised pretext tasks (e.g. **SimSiam**, **Barlow Twins**).

### Key Features:
- **Batch Experiment Runner**: Pass a list of configuration files to `CONFIG_PATHS` to queue and run multiple experiments sequentially.
- **Isolated Outputs**: Each configuration writes to its own distinct run folder (e.g., `runs/pretext/pretext_multitask_convnext_barlow_v2/`).
- **Live Drive Sync**: Checkpoints (`last.pt`), outputs, and metrics are automatically synced to Google Drive (`--sync-dir`).
- **Summary Comparison Table**: At the end of execution, collates Top-1, Top-5, per-head disease/crop accuracy, and Harmonic Means into an interactive comparison table.


In [1]:
# ================= Step 1: Configuration & Experiment Parameters ================
import os
from pathlib import Path

# --- Model & Training Configurations ---
# Provide either a single path (str) or a list of paths (list[str]).
# Each configuration runs as an individual experiment with its own run folder.
CONFIG_PATHS = [
    "config/pretext/pretext_multitask_convnext_barlow_v2.json",
    "config/pretext/pretext_multitask_convnext_barlow_v3_wide.json",
]
if isinstance(CONFIG_PATHS, str):
    CONFIG_PATHS = [CONFIG_PATHS]

RESUME = True  # Set to True to resume from last saved checkpoint if interrupted

# Command-line overrides applied via --set key.path=value to every run in the queue
OVERRIDES = [
    "runtime.num_workers=2",                # Recommended for standard Colab vCPUs
    "runtime.checkpoint_every_n_epochs=5",  # Save checkpoint every 5 epochs
]

# --- Google Drive & Sync Paths ---
GDRIVE_PROJECT = Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment")
GDRIVE_DATASET_7Z = GDRIVE_PROJECT / "DPD_Small.7z"
GDRIVE_RUNS = GDRIVE_PROJECT / "runs"
SYNC_DIR = str(GDRIVE_RUNS)

# --- Fast Local Storage Paths (/content/) ---
LOCAL_ARCHIVE = Path("/content/DPD_Small.7z")
EXTRACT_DEST = Path("/content/dataset")

print(f"Queued {len(CONFIG_PATHS)} experiment(s) to run:")
for i, p in enumerate(CONFIG_PATHS, 1):
    print(f"  [{i}] {p}")
print(f"Project Dir:   {GDRIVE_PROJECT}")
print(f"Sync Dir:      {SYNC_DIR}")
print(f"Resume:        {RESUME}")
print(f"Overrides:     {OVERRIDES}")


Queued 2 experiment(s):
  [1] config/pretext/pretext_multitask_convnext_simsiam.json
  [2] config/pretext/pretext_multitask_convnext_barlow.json
Project Dir:   /content/drive/MyDrive/Nat_AML_DPD_group_assignment
Sync Dir:      /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs
Resume:        True
Overrides:     ['runtime.num_workers=2', 'runtime.checkpoint_every_n_epochs=5']
Gallery Grid:  4 cols x 3 rows


In [2]:
# Step 2: Mount Google Drive & Verify Project Folder
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Verify paths configured in Step 1 exist
assert GDRIVE_PROJECT.exists(), f"Google Drive folder not found: {GDRIVE_PROJECT}"
assert GDRIVE_DATASET_7Z.exists(), f"Dataset archive not found at: {GDRIVE_DATASET_7Z}"
GDRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted and verified successfully!")
print(f"Project folder: {GDRIVE_PROJECT}")
print(f"Dataset archive: {GDRIVE_DATASET_7Z}")
print(f"Persistent runs folder: {GDRIVE_RUNS}")

Mounted at /content/drive
Google Drive mounted and verified successfully!
Project folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment
Dataset archive: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z
Persistent runs folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs


In [3]:
# Step 3: Extract Dataset to Fast Local Colab SSD (/content/)
import shutil
from pathlib import Path
import os

EXTRACT_DEST.mkdir(parents=True, exist_ok=True)

# 1. Copy .7z from Google Drive to local SSD (if not already copied)
if not LOCAL_ARCHIVE.exists():
    print(f"Copying {GDRIVE_DATASET_7Z} to local SSD ({LOCAL_ARCHIVE})...")
    shutil.copy2(GDRIVE_DATASET_7Z, LOCAL_ARCHIVE)
    print("Copy complete!")
else:
    print(f"Local archive {LOCAL_ARCHIVE} already present.")

# 2. Extract using 7z if CSVs are not yet present
def find_dpd_root(search_dir: Path) -> Path | None:
    for candidate in [search_dir, search_dir / "DPD_Small", *search_dir.glob("**/DPD_Small*")]:
        if candidate.is_dir() and (candidate / "DPD_Small_train.csv").exists():
            return candidate
    return None

dpd_root = find_dpd_root(EXTRACT_DEST)
if dpd_root is None:
    print("Extracting dataset with 7z (fast multithreaded)...", flush=True)
    !7z x -y /content/DPD_Small.7z -o/content/dataset
    dpd_root = find_dpd_root(EXTRACT_DEST)

assert dpd_root is not None, f"Could not find DPD_Small_train.csv inside {EXTRACT_DEST}!"

# 3. Verify all dataset components exist
assert (dpd_root / "DPD_Small_train.csv").exists(), f"Missing train CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_seen.csv").exists(), f"Missing test seen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_unseen.csv").exists(), f"Missing test unseen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_dataset").exists(), f"Missing DPD_Small_dataset folder in {dpd_root}"

# 4. Export DPD_ROOT environment variable (read by dpd/data.py)
os.environ["DPD_ROOT"] = str(dpd_root)
print(f"\nDataset verified successfully!")
print(f"DPD_ROOT set to: {os.environ['DPD_ROOT']}")

Copying /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z to local SSD (/content/DPD_Small.7z)...
Copy complete!
Extracting dataset with 7z (fast multithreaded)...

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:12 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content                  1 file, 3399045528 bytes (3242 MiB)

Extracting archive: /content/DPD_Small.7z
--
Path = /content/DPD_Small.7z
Type = 7z
Physical Size = 3399045528
Headers Size = 107901
Method = LZMA2:25
Solid = +
Blocks = 1

      0%      1%      2%      3%      4%      5%      6%      7%      8%      9%     10%     11%     12%     13%     15%     16%     17%     18%     19%     20%     21%     22%     23%     24%     25%     26%     27%     28%     29%     30%     31%     32%     33%     34%     35%     36%     37%     38%     39%     40%     41% 393 - DPD_Small_dataset/CC_coffee_leaf_miner_226.jp                                    

In [4]:
# Step 4: Install Required Python Packages
!apt-get install -y -qq
!pip install -q timm pandas pillow tqdm matplotlib
!nvidia-smi


Sun Nov 15 07:46:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
# Step 5: Run Pretext Multi-Task Training Across All Configs
import subprocess
import sys
import time
import os
from pathlib import Path

# Automatically locate the directory that contains the 'train' and 'dpd' modules
candidates = [
    Path("/content"),
    Path.cwd(),
    Path("/content/Out-Of-Distribution-Plant-Disease-Identification"),
    Path("/"),
    Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment"),
]
project_root = None
for c in candidates:
    if (c / "train" / "train_pretext_multitask.py").exists() or (c / "train").is_dir():
        project_root = c.resolve()
        break

if project_root is None:
    matches = list(Path("/content").glob("**/train_pretext_multitask.py"))
    project_root = matches[0].parent.parent.resolve() if matches else Path("/content").resolve()

print(f"Project root located at: {project_root}")
os.chdir(str(project_root))

# Ensure PYTHONPATH contains project_root so 'train' and 'dpd' are always found
env = os.environ.copy()
env["PYTHONPATH"] = f"{str(project_root)}:{env.get('PYTHONPATH', '')}"

print(f"Queued {len(CONFIG_PATHS)} experiment(s) to run:")
for i, cp in enumerate(CONFIG_PATHS, 1):
    print(f"  [{i}] {cp}")
print("=" * 70)

for idx, config_path in enumerate(CONFIG_PATHS, 1):
    print("\n" + "=" * 70)
    print(f"[{idx}/{len(CONFIG_PATHS)}] Starting Experiment: {config_path}")
    print("=" * 70)

    # Resolve config path
    cfg_target = config_path if Path(config_path).is_absolute() else (project_root / config_path)
    if not cfg_target.exists():
        matches = list(project_root.glob(f"**/{Path(config_path).name}"))
        if matches:
            cfg_target = matches[0]

    cmd = [
        sys.executable,
        "-m", "train.train_pretext_multitask",
        "--config", str(cfg_target),
        "--sync-dir", SYNC_DIR,
    ]
    if OVERRIDES:
        cmd.append("--set")
        cmd.extend(OVERRIDES)
    if RESUME:
        cmd.append("--resume")

    print("Starting training execution:")
    print(" ".join(cmd))
    print("-" * 70)

    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=str(project_root),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="", flush=True)

    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Experiment {config_path} exited with error code {proc.returncode}")

    elapsed = (time.time() - t0) / 60.0
    print(f"\nCompleted [{idx}/{len(CONFIG_PATHS)}] {config_path} in {elapsed:.1f} minutes!")

print("\nAll queued experiments finished successfully and outputs are synced to Google Drive!")

Project root located at: /content
Queued 2 experiment(s) to run:
  [1] config/pretext/pretext_multitask_convnext_simsiam.json
  [2] config/pretext/pretext_multitask_convnext_barlow.json

[1/2] Starting Experiment: config/pretext/pretext_multitask_convnext_simsiam.json
Starting training execution:
/usr/bin/python3 -m train.train_pretext_multitask --config /content/config/pretext/pretext_multitask_convnext_simsiam.json --sync-dir /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs --set runtime.num_workers=2 runtime.checkpoint_every_n_epochs=5 --resume
----------------------------------------------------------------------
Applied overrides:
  runtime.num_workers=2
  runtime.checkpoint_every_n_epochs=5
Live synchronization enabled -> /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/pretext/pretext_multitask_convnext_simsiam

resize to 256px: 100%|██████████| 11448/11448 [00:20<00:00, 555.32it/s]
Gamma tuning disabled (gamma_grid=0): evaluating uncalibrated (gamma=0.0) across 

In [6]:
# Step 6: Multi-Experiment Comparison Summary Table (Calibrated & Uncalibrated)
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

# Check both current queue and existing pretext experiment configs
all_candidate_configs = [
    "config/pretext/pretext_multitask_convnext_simsiam.json",
    "config/pretext/pretext_multitask_convnext_barlow.json",
    "config/pretext/pretext_multitask_convnext_barlow_v2.json",
    "config/pretext/pretext_multitask_convnext_barlow_v3_wide.json",
]
for cp in CONFIG_PATHS:
    if cp not in all_candidate_configs:
        all_candidate_configs.append(cp)

records = []
for config_path in all_candidate_configs:
    cfg_target = config_path if Path(config_path).is_absolute() else (project_root / config_path)
    if not cfg_target.exists():
        matches = list(project_root.glob(f"**/{Path(config_path).name}"))
        if matches:
            cfg_target = matches[0]
        else:
            continue
    try:
        with open(cfg_target, "r") as f:
            cfg = json.load(f)
    except Exception:
        continue

    run_name = cfg.get("version", Path(config_path).stem)
    rel_out = Path(cfg["output_dir"])
    local_dir = rel_out if rel_out.is_absolute() else (project_root / rel_out)
    gdrive_dir = GDRIVE_RUNS / Path(*rel_out.parts[1:]) if len(rel_out.parts) > 1 and rel_out.parts[0] == "runs" else (GDRIVE_RUNS / rel_out)
    out_dir = gdrive_dir if gdrive_dir.exists() else local_dir
    metrics_file = out_dir / "metrics.json"
    metrics_gamma_file = out_dir / "metrics_gamma.json"

    if metrics_file.exists():
        with open(metrics_file, "r") as f:
            m_uncal = json.load(f)
        m_cal = m_uncal
        if metrics_gamma_file.exists():
            try:
                with open(metrics_gamma_file, "r") as f:
                    m_cal = json.load(f)
            except Exception:
                pass

        seen_cal = m_cal.get("test_seen", {})
        unseen_cal = m_cal.get("test_unseen", {})
        unseen_uncal = m_uncal.get("test_unseen", {})
        pretext_type = cfg.get("loss", {}).get("pretext", {}).get("type", "none")
        lambda_pretext = cfg.get("loss", {}).get("pretext", {}).get("lambda_pretext", 0.0)
        gamma_val = m_cal.get("gamma", 0.0)

        records.append({
            "Experiment": run_name,
            "Pretext Task": f"{pretext_type} (λ={lambda_pretext})",
            "Epoch": m_cal.get("chosen_epoch") or m_cal.get("selected_epoch"),
            "Tuned Gamma": round(gamma_val, 2) if gamma_val is not None else 0.0,
            "Calib HM Top-1": round(m_cal.get("harmonic_mean_top1", 0.0), 4),
            "Calib Seen Top-1": round(seen_cal.get("top1", 0.0), 4),
            "Calib Unseen Top-1": round(unseen_cal.get("top1", 0.0), 4),
            "Calib Unseen Disease": round(unseen_cal.get("disease_top1", 0.0), 4),
            "Uncalib HM (γ=0)": round(m_uncal.get("harmonic_mean_top1", 0.0), 4),
            "Uncalib Unseen Top-1": round(unseen_uncal.get("top1", 0.0), 4),
            "Output Dir": str(out_dir),
        })
    elif config_path in CONFIG_PATHS:
        records.append({
            "Experiment": run_name,
            "Pretext Task": cfg.get("loss", {}).get("pretext", {}).get("type", "none"),
            "Calib HM Top-1": "In Progress / Queued",
            "Output Dir": str(out_dir),
        })

df_summary = pd.DataFrame(records)
print("\n" + "=" * 105)
print("MULTI-EXPERIMENT PERFORMANCE SUMMARY TABLE (Calibrated & Uncalibrated)")
print("=" * 105)
display(df_summary)

summary_csv = GDRIVE_RUNS / "pretext_comparison_summary.csv"
df_summary.to_csv(summary_csv, index=False)
print(f"\nSummary table saved to: {summary_csv}")



MULTI-EXPERIMENT PERFORMANCE SUMMARY TABLE (Uncalibrated gamma=0.0)


,Experiment,Pretext Task,Epoch,Gamma,HM Top-1,HM Top-5,Seen Top-1,Seen Disease,Seen Crop,Unseen Top-1,Unseen Disease,Unseen Crop,Output Dir
0,pretext_multitask_convnext_simsiam,simsiam (λ=0.5),16,0.0,0.0647,0.4360,0.7335,0.747,0.942,0.0338,0.0925,0.7714,/content/drive/MyDrive/Nat_AML_DPD_group_assig...
1,pretext_multitask_convnext_barlow,barlow_twins (λ=0.5),7,0.0,0.1069,0.4827,0.6770,0.694,0.919,0.0580,0.1070,0.7065,/content/drive/MyDrive/Nat_AML_DPD_group_assig...



Summary table saved to: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/pretext_comparison_summary.csv
